# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

c:\Users\krist\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [9]:
# Write your code below.
dd_px["Close_lag_1"] = dd_px["Close"].shift(1)
dd_px["Adj_Close_lag_1"] = dd_px["Adj Close"].shift(1)

dd_feat = dd_px.assign(
    Returns = lambda x: x["Close"]/x["Close_lag_1"] - 1,
    hi_lo_range = lambda x: x['High'] - x['Low']
)

dd_feat.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag,Adj_Close_lag,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
ticker,,,,,,,,,,,,,,,
ACN,2017-07-05,124.000000,124.779999,123.989998,124.050003,118.719963,1328600.0,ACN.csv,2017,NaN,NaN,NaN,NaN,NaN,0.790001
ACN,2017-07-06,123.349998,123.629997,122.029999,122.940002,117.657669,1938100.0,ACN.csv,2017,124.050003,118.719963,124.050003,118.719963,-0.008948,1.599998
ACN,2017-07-07,123.550003,124.750000,123.070000,124.209999,118.873100,1889400.0,ACN.csv,2017,122.940002,117.657669,122.940002,117.657669,0.010330,1.680000
ACN,2017-07-10,124.050003,124.370003,123.589996,124.059998,118.729546,1301500.0,ACN.csv,2017,124.209999,118.873100,124.209999,118.873100,-0.001208,0.780006
ACN,2017-07-11,124.059998,124.099998,123.199997,123.849998,118.528572,1379800.0,ACN.csv,2017,124.059998,118.729546,124.059998,118.729546,-0.001693,0.900002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2003-12-24,7.650000,7.960000,7.500000,7.740000,7.740000,236300.0,ZIXI.csv,2003,7.620000,7.620000,7.620000,7.620000,0.015748,0.460000
ZIXI,2003-12-26,7.900000,8.130000,7.800000,8.060000,8.060000,144200.0,ZIXI.csv,2003,7.740000,7.740000,7.740000,7.740000,0.041344,0.330000
ZIXI,2003-12-29,8.000000,8.100000,7.810000,7.880000,7.880000,245400.0,ZIXI.csv,2003,8.060000,8.060000,8.060000,8.060000,-0.022333,0.290000


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [14]:
# Write your code below.

df_feat = dd_feat.compute()

df_feat["Returns_Move_Avg"] = df_feat["Returns"].rolling(10).mean()

df_feat.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag,Adj_Close_lag,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,Returns_Move_Avg
ticker,,,,,,,,,,,,,,,,
ACN,2017-07-05,124.000000,124.779999,123.989998,124.050003,118.719963,1328600.0,ACN.csv,2017,NaN,NaN,NaN,NaN,NaN,0.790001,NaN
ACN,2017-07-06,123.349998,123.629997,122.029999,122.940002,117.657669,1938100.0,ACN.csv,2017,124.050003,118.719963,124.050003,118.719963,-0.008948,1.599998,NaN
ACN,2017-07-07,123.550003,124.750000,123.070000,124.209999,118.873100,1889400.0,ACN.csv,2017,122.940002,117.657669,122.940002,117.657669,0.010330,1.680000,NaN
ACN,2017-07-10,124.050003,124.370003,123.589996,124.059998,118.729546,1301500.0,ACN.csv,2017,124.209999,118.873100,124.209999,118.873100,-0.001208,0.780006,NaN
ACN,2017-07-11,124.059998,124.099998,123.199997,123.849998,118.528572,1379800.0,ACN.csv,2017,124.059998,118.729546,124.059998,118.729546,-0.001693,0.900002,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it was not necessary to convert to pandas to calculate the moving average return because the operation is also supported in Dask.

No, it would not have been better to do it in Dask because Dask is more suited for larger datasets and our dataset is not considered large. Pandas is more suited to handle smaller datasets (like the one we have).

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.